# Real World: Czech Municipalities

Is the population of Czech municipalities a **power law** or a **normal distribution**?

Data: ČSÚ, [Počet obyvatel v obcích k 1. 1. 2025](https://csu.gov.cz/produkty/pocet-obyvatel-v-obcich-rlm0s92pwn).

In [1]:
import numpy as np
import polars as pl
import powerlaw
from scipy import stats
from utils.graph import (
    plot_population_histogram,
    plot_rank_survival,
    plot_sample_parameters,
    plot_tail_comparison,
)

## Data

In [2]:
COLUMNS = [
    "district_code",
    "municipality_code",
    "name",
    "population",
    "men",
    "women",
    "average_age",
    "average_age_men",
    "average_age_women",
]

df = pl.read_excel(
    "data/pocet_obyvatel_obce_2025.xlsx", has_header=False, read_options={"skip_rows": 6}
)
municipalities = (
    df.rename(dict(zip(df.columns, COLUMNS)))
    .filter(pl.col("municipality_code").is_not_null(), pl.col("population") > 0)
    .sort("population", descending=True)
)
population = municipalities["population"].to_numpy()

/var/folders/kk/g2nkcl3s7qx6rk7t5xxcqp6w0000gn/T/ipykernel_70234/369267735.py:13: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df = pl.read_excel(


In [3]:
municipalities.head()

district_code,municipality_code,name,population,men,women,average_age,average_age_men,average_age_women
str,i64,str,i64,i64,i64,f64,f64,f64
"""CZ0100""",554782,"""Praha""",1397880,679162,718718,41.864895,40.356198,43.290559
"""CZ0642""",582786,"""Brno""",402739,195615,207124,42.74713,40.943565,44.450479
"""CZ0806""",554821,"""Ostrava""",283187,137397,145790,43.902282,42.020412,45.675815
"""CZ0323""",554791,"""Plzeň""",187928,90848,97080,43.02924,41.36342,44.588123
"""CZ0513""",563889,"""Liberec""",108090,51989,56101,42.769535,41.096761,44.3197


In [4]:
municipalities["population"].describe(percentiles=[0.25, 0.5, 0.75, 0.99])

statistic,value
str,f64
"""count""",6254.0
"""null_count""",0.0
"""mean""",1744.403582
"""std""",19473.077704
"""min""",16.0
"""25%""",224.0
"""50%""",453.0
"""75%""",987.0
"""99%""",20002.0


In [5]:
plot_population_histogram(population)

## Models Fitting using MLE

- **normal**: $\hat\mu$, $\hat\sigma$ = sample mean and std
- **power law**: MLE $\hat\alpha$ for every candidate $x_{\min}$, keep the one with the smallest KS distance

In [6]:
def power_law_quantile(q: np.ndarray, alpha: float, x_min: float) -> np.ndarray:
    # q = P(X >= x | X >= x_min)
    return x_min * q ** (-1 / (alpha - 1))


def power_law_mean(alpha: float, x_min: float, x_max: float = np.inf) -> float:
    k = (alpha - 1) / (x_min ** (1 - alpha) - x_max ** (1 - alpha))
    return k * (x_min ** (2 - alpha) - x_max ** (2 - alpha)) / (alpha - 2)


fit = powerlaw.Fit(population, discrete=True, verbose=False)
alpha, x_min = fit.power_law.alpha, fit.power_law.xmin
tail = population[population >= x_min]
mu, sigma = population.mean(), population.std()

print(
    f"power law: α = {alpha:.3f} ± {fit.power_law.sigma:.3f}, x_min = {x_min:,.0f}, tail = {len(tail) / len(population):.1%} of municipalities, {tail.sum() / population.sum():.1%} of people"
)
print(
    f"normal:    μ = {mu:,.0f}, σ = {sigma:,.0f}, P(X < 0) = {stats.norm.cdf(0, mu, sigma):.0%}, P(X ≥ x_min) = {stats.norm.sf(x_min, mu, sigma):.0%}"
)

power law: α = 2.093 ± 0.031, x_min = 1,264, tail = 19.3% of municipalities, 80.1% of people
normal:    μ = 1,744, σ = 19,472, P(X < 0) = 46%, P(X ≥ x_min) = 51%


In [7]:
plot_rank_survival(population, municipalities["name"].to_list(), alpha, x_min)

## Comparison on the Tail

In [8]:
def tail_normal(mu: float, sigma: float, x_min: float):
    return stats.truncnorm((x_min - mu) / sigma, np.inf, mu, sigma)


normal = tail_normal(mu, sigma, x_min)
models = {
    "power law": {
        "sf": lambda x: (x / x_min) ** (1 - alpha),
        "isf": lambda q: power_law_quantile(q, alpha, x_min),
        "logpdf": fit.power_law.loglikelihoods,
        "total": len(tail) * power_law_mean(alpha, x_min, tail.max()),
        "log10_sf_max": (1 - alpha) * np.log10(tail.max() / x_min),
    },
    "normal": {
        "sf": normal.sf,
        "isf": normal.isf,
        "logpdf": normal.logpdf,
        "total": len(tail) * normal.mean(),
        "log10_sf_max": normal.logsf(tail.max()) / np.log(10),
    },
}

pl.DataFrame(
    {
        "": ["observed", *models],
        "tail median": [np.median(tail), *(m["isf"](0.5) for m in models.values())],
        "tail population [M]": [tail.sum() / 1e6, *(m["total"] / 1e6 for m in models.values())],
        "log-likelihood": [None, *(m["logpdf"](tail).sum() for m in models.values())],
        "log10 P(X ≥ Prague | tail)": [None, *(m["log10_sf_max"] for m in models.values())],
    }
).with_columns(pl.selectors.float().round(2))

,tail median,tail population [M],log-likelihood,log10 P(X ≥ Prague | tail)
str,f64,f64,f64,f64
"""observed""",2347.0,8.74,null,null
"""power law""",2382.54,8.6,-10840.83,-3.33
"""normal""",14577.77,20.52,-15352.41,-1118.34


In [9]:
tail_frame = municipalities.filter(pl.col("population") >= x_min).sort("population")
plot_tail_comparison(
    tail_frame["population"].to_numpy(), tail_frame["name"].to_list(), x_min, models
)

## Random Sample Model Fitting

- sample sizes $n \in [50, 6\,254]$
- 40 random samples each, refit the power law

In [10]:
def fit_sample(sample: np.ndarray) -> dict:
    fit = powerlaw.Fit(sample, discrete=True, verbose=False)
    return {"alpha": fit.power_law.alpha, "x_min": fit.power_law.xmin}


rng = np.random.default_rng(42)
sample_sizes = np.unique(np.round(np.geomspace(50, len(population), 12)).astype(int))
experiment = pl.DataFrame(
    [
        {"sample_size": n, **fit_sample(rng.choice(population, n, replace=False))}
        for n in sample_sizes
        for _ in range(40)
    ]
)
full_data = fit_sample(population)

In [11]:
plot_sample_parameters(experiment, full_data)